# Test `asinstata=True` vs `asinstata=False`

This notebook runs all 6 applied examples from the paper with both settings and compares results side-by-side.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from did_multiplegt_stat import did_multiplegt_stat

# Load data
df = pd.read_stata('gazoline_did_multiplegt_stat.dta')
df_g = pd.read_stata('gentzkowetal_didtextbook.dta')
print(f'Gazoline: {df.shape}')
print(f'Gentzkow: {df_g.shape}')

Gazoline: (2064, 354)
Gentzkow: (16872, 16)


In [2]:
def extract_estimates(result):
    """Extract AS and WAS estimates from a result dict."""
    out = {}
    tbl = result.get('results', {}).get('table', None)
    if tbl is None:
        tbl = result.get('results', {}).get('table_placebo_1', None)
    if isinstance(tbl, pd.DataFrame):
        for idx in tbl.index:
            out[idx] = tbl.loc[idx, 'Estimate'] if 'Estimate' in tbl.columns else tbl.iloc[tbl.index.get_loc(idx), 0]
    # Also check for direct table
    tbl2 = result.get('results', {}).get('table', None)
    if isinstance(tbl2, pd.DataFrame):
        for idx in tbl2.index:
            val = tbl2.iloc[tbl2.index.get_loc(idx), 0]
            out[idx] = val
    # Placebo tables
    for pl in range(1, 5):
        pl_tbl = result.get('results', {}).get(f'table_placebo_{pl}', None)
        if isinstance(pl_tbl, pd.DataFrame):
            for idx in pl_tbl.index:
                out[f'Plac{pl}_{idx}'] = pl_tbl.iloc[pl_tbl.index.get_loc(idx), 0]
    return out


def compare_results(name, res_true, res_false):
    """Build a comparison DataFrame."""
    est_t = extract_estimates(res_true)
    est_f = extract_estimates(res_false)
    all_keys = sorted(set(list(est_t.keys()) + list(est_f.keys())))
    rows = []
    for k in all_keys:
        vt = est_t.get(k, np.nan)
        vf = est_f.get(k, np.nan)
        if pd.notna(vt) and pd.notna(vf) and vt != 0:
            pct = abs(vt - vf) / abs(vt) * 100
        else:
            pct = np.nan
        rows.append({'Estimator': k, 'asinstata=True': vt, 'asinstata=False': vf,
                     'Diff': vt - vf if pd.notna(vt) and pd.notna(vf) else np.nan,
                     'Diff%': pct})
    comp = pd.DataFrame(rows)
    comp.index = comp['Estimator']
    comp = comp.drop(columns='Estimator')
    print(f'\n{"="*60}')
    print(f'  {name}')
    print(f'{"="*60}')
    display(comp)
    return comp

## V.01: AOSS+WAOSS with controls (lngca ~ tau)

In [3]:
print('Running V.01 asinstata=True...')
r01_t = did_multiplegt_stat(df, 'lngca', 'id', 'year', 'tau',
                            order=1, aoss_vs_waoss=True, controls=['lngpinc'],
                            asinstata=True)
print('Running V.01 asinstata=False...')
r01_f = did_multiplegt_stat(df, 'lngca', 'id', 'year', 'tau',
                            order=1, aoss_vs_waoss=True, controls=['lngpinc'],
                            asinstata=False)
comp01 = compare_results('V.01: AOSS+WAOSS, controls=[lngpinc], Y=lngca', r01_t, r01_f)

Running V.01 asinstata=True...


Running V.01 asinstata=False...



  V.01: AOSS+WAOSS, controls=[lngpinc], Y=lngca


,asinstata=True,asinstata=False,Diff,Diff%
Estimator,,,,
AS,-0.001384,-0.001383,-3.166807e-07,0.022885
IV-WAS,NaN,NaN,NaN,NaN
WAS,-0.003756,-0.003754,-2.077597e-06,0.055307
as_10,-0.002407,-0.002407,-6.701975e-08,0.002785
as_11,0.003483,0.003483,-4.592609e-08,0.001318
...,...,...,...,...
was_5,-0.003830,-0.003830,-7.688423e-08,0.002007
was_6,0.008187,0.008187,9.709137e-09,0.000119
was_7,-0.006903,-0.006903,-9.382879e-09,0.000136


## V.02: AOSS+WAOSS with controls (lngpinc ~ tau)

In [4]:
print('Running V.02 asinstata=True...')
r02_t = did_multiplegt_stat(df, 'lngpinc', 'id', 'year', 'tau',
                            order=1, aoss_vs_waoss=True, controls=['lngpinc'],
                            asinstata=True)
print('Running V.02 asinstata=False...')
r02_f = did_multiplegt_stat(df, 'lngpinc', 'id', 'year', 'tau',
                            order=1, aoss_vs_waoss=True, controls=['lngpinc'],
                            asinstata=False)
comp02 = compare_results('V.02: AOSS+WAOSS, controls=[lngpinc], Y=lngpinc', r02_t, r02_f)

Running V.02 asinstata=True...


Running V.02 asinstata=False...



  V.02: AOSS+WAOSS, controls=[lngpinc], Y=lngpinc


,asinstata=True,asinstata=False,Diff,Diff%
Estimator,,,,
AS,0.001101,0.001100,1.073256e-06,0.097456
IV-WAS,NaN,NaN,NaN,NaN
WAS,0.005660,0.005671,-1.149126e-05,0.203033
as_10,0.001348,0.001348,6.076407e-07,0.045063
as_11,-0.000881,-0.000881,-1.158155e-07,0.013146
...,...,...,...,...
was_5,0.003319,0.003319,-1.477511e-07,0.004451
was_6,0.026107,0.026107,1.130890e-07,0.000433
was_7,0.021391,0.021390,1.837890e-07,0.000859


## V.03: IV-WAOSS

In [5]:
print('Running V.03 asinstata=True...')
r03_t = did_multiplegt_stat(df, 'lngca', 'id', 'year', 'lngpinc',
                            Z='tau', estimator='ivwaoss', order=1,
                            controls=['lngpinc'], bootstrap=5, seed=1,
                            asinstata=True)
print('Running V.03 asinstata=False...')
r03_f = did_multiplegt_stat(df, 'lngca', 'id', 'year', 'lngpinc',
                            Z='tau', estimator='ivwaoss', order=1,
                            controls=['lngpinc'], bootstrap=5, seed=1,
                            asinstata=False)
comp03 = compare_results('V.03: IV-WAOSS, bootstrap=5', r03_t, r03_f)

Running V.03 asinstata=True...
                              First stage estimation


                              Reduced form estimation


Bootstrap (5 replications)...


Running V.03 asinstata=False...
                              First stage estimation


                              Reduced form estimation


Bootstrap (5 replications)...



  V.03: IV-WAOSS, bootstrap=5


,asinstata=True,asinstata=False,Diff,Diff%
Estimator,,,,
AS,NaN,NaN,NaN,NaN
IV-WAS,-0.663708,-0.661997,-0.001711,0.257817
WAS,NaN,NaN,NaN,NaN
as_10,NaN,NaN,NaN,NaN
as_11,NaN,NaN,NaN,NaN
...,...,...,...,...
was_5,NaN,NaN,NaN,NaN
was_6,NaN,NaN,NaN,NaN
was_7,NaN,NaN,NaN,NaN


## V.04: WAOSS on placebo sample (lngca)

In [6]:
print('Running V.04 asinstata=True...')
r04_t = did_multiplegt_stat(df, 'lngca', 'id', 'year', 'tau',
                            estimator='waoss', order=1, controls=['lngpinc'],
                            on_placebo_sample=True, asinstata=True)
print('Running V.04 asinstata=False...')
r04_f = did_multiplegt_stat(df, 'lngca', 'id', 'year', 'tau',
                            estimator='waoss', order=1, controls=['lngpinc'],
                            on_placebo_sample=True, asinstata=False)
comp04 = compare_results('V.04: WAOSS, on_placebo_sample, Y=lngca', r04_t, r04_f)

Running V.04 asinstata=True...


Running V.04 asinstata=False...



  V.04: WAOSS, on_placebo_sample, Y=lngca


,asinstata=True,asinstata=False,Diff,Diff%
Estimator,,,,
AS,NaN,NaN,NaN,NaN
IV-WAS,NaN,NaN,NaN,NaN
WAS,-0.002944,-0.002944,2.413011e-07,0.008197
as_10,NaN,NaN,NaN,NaN
as_11,NaN,NaN,NaN,NaN
...,...,...,...,...
was_5,0.002601,0.002601,-2.310135e-08,0.000888
was_6,0.009703,0.009703,7.713804e-08,0.000795
was_7,-0.004311,-0.004311,7.521389e-08,0.001745


## V.05: WAOSS on placebo sample (lngpinc)

In [7]:
print('Running V.05 asinstata=True...')
r05_t = did_multiplegt_stat(df, 'lngpinc', 'id', 'year', 'tau',
                            estimator='waoss', order=1, controls=['lngpinc'],
                            on_placebo_sample=True, asinstata=True)
print('Running V.05 asinstata=False...')
r05_f = did_multiplegt_stat(df, 'lngpinc', 'id', 'year', 'tau',
                            estimator='waoss', order=1, controls=['lngpinc'],
                            on_placebo_sample=True, asinstata=False)
comp05 = compare_results('V.05: WAOSS, on_placebo_sample, Y=lngpinc', r05_t, r05_f)

Running V.05 asinstata=True...


Running V.05 asinstata=False...



  V.05: WAOSS, on_placebo_sample, Y=lngpinc


,asinstata=True,asinstata=False,Diff,Diff%
Estimator,,,,
AS,NaN,NaN,NaN,NaN
IV-WAS,NaN,NaN,NaN,NaN
WAS,0.005041,0.005041,2.254419e-07,0.004472
as_10,NaN,NaN,NaN,NaN
as_11,NaN,NaN,NaN,NaN
...,...,...,...,...
was_5,0.011936,0.011936,2.250055e-07,0.001885
was_6,0.025484,0.025484,3.682936e-08,0.000145
was_7,0.023670,0.023671,-1.253637e-06,0.005296


## VI.01: Gentzkow exact_match

In [8]:
print('Running VI.01 asinstata=True...')
r06_t = did_multiplegt_stat(df_g, 'prestout', 'cnty90', 'year', 'numdailies',
                            placebo=1, exact_match=True, asinstata=True)
print('Running VI.01 asinstata=False...')
r06_f = did_multiplegt_stat(df_g, 'prestout', 'cnty90', 'year', 'numdailies',
                            placebo=1, exact_match=True, asinstata=False)
comp06 = compare_results('VI.01: Gentzkow, exact_match, placebo=1', r06_t, r06_f)

Running VI.01 asinstata=True...


Running VI.01 asinstata=False...



  VI.01: Gentzkow, exact_match, placebo=1


,asinstata=True,asinstata=False,Diff,Diff%
Estimator,,,,
AS,0.006072,0.006072,0.0,0.0
IV-WAS,NaN,NaN,NaN,NaN
Plac1_Placebo_1,-0.001128,-0.001128,0.0,0.0
Plac1_Placebo_1_ivwaoss,0.000000,0.000000,0.0,NaN
Plac1_Placebo_1_waoss,-0.000013,-0.000013,0.0,0.0
WAS,0.005719,0.005719,0.0,0.0
as_10,0.014020,0.014020,0.0,0.0
as_11,-0.002290,-0.002290,0.0,0.0
as_12,0.003716,0.003716,0.0,0.0


## Summary

In [9]:
print('\n' + '='*70)
print('  SUMMARY: max |Diff%| per example')
print('='*70)
summaries = [
    ('V.01', comp01), ('V.02', comp02), ('V.03', comp03),
    ('V.04', comp04), ('V.05', comp05), ('VI.01', comp06)
]
for name, comp in summaries:
    max_pct = comp['Diff%'].max()
    print(f'  {name}: max Diff% = {max_pct:.4f}%')


  SUMMARY: max |Diff%| per example
  V.01: max Diff% = 83.2985%
  V.02: max Diff% = 12.0020%
  V.03: max Diff% = 19.6879%
  V.04: max Diff% = 2.6117%
  V.05: max Diff% = 7.5461%
  VI.01: max Diff% = 0.0000%
